In [3]:
import pandas as pd

In [4]:
import pandas as pd
from pathlib import Path

# Path to the ONS file — adjust filename if yours is different
ons_file = Path("data/raw/ons/myebtablesuk20112024.xlsx")
# Load only the MYEB4 sheet
# skiprows=1 skips the title row so row 2 becomes the header
pop_raw = pd.read_excel(
    ons_file,
    sheet_name="MYEB4",
    skiprows=1
)

print(f"Shape: {pop_raw.shape}")
print(f"Columns: {list(pop_raw.columns)}")
print(f"\nFirst 5 rows:")
pop_raw.head()

Shape: (1274, 18)
Columns: ['Code', 'Name', 'sex', 'age', 'population_2011', 'population_2012', 'population_2013', 'population_2014', 'population_2015', 'population_2016', 'population_2017', 'population_2018', 'population_2019', 'population_2020', 'population_2021', 'population_2022', 'population_2023', 'population_2024']

First 5 rows:


,Code,Name,sex,age,population_2011,population_2012,population_2013,population_2014,population_2015,population_2016,population_2017,population_2018,population_2019,population_2020,population_2021,population_2022,population_2023,population_2024
0,K02000001,UNITED KINGDOM,f,0,390106,394999,384448,376959,375454,378033,368424,358986,349297,340787,329463,338736,326859,326016
1,K02000001,UNITED KINGDOM,f,1,385998,392160,396377,385952,378269,376655,378418,367931,358796,350191,343078,336812,348757,336137
2,K02000001,UNITED KINGDOM,f,2,382279,387647,393532,397402,386736,378961,376348,377822,367177,357453,350466,350243,346817,357634
3,K02000001,UNITED KINGDOM,f,3,385930,383826,389048,394885,398146,387365,378648,375670,377466,366025,356632,357740,360469,355532
4,K02000001,UNITED KINGDOM,f,4,372956,387189,385003,390256,395912,398621,387091,377951,375430,376606,365695,363348,367540,369036


In [5]:
pop_raw["Name"].unique()

<StringArray>
[   'UNITED KINGDOM',     'GREAT BRITAIN', 'ENGLAND AND WALES',
           'ENGLAND',             'WALES',          'SCOTLAND',
  'NORTHERN IRELAND']
Length: 7, dtype: str

In [6]:
countries = pop_raw[(pop_raw["Name"] == "ENGLAND") | (pop_raw["Name"] == "SCOTLAND")]

In [7]:
countries["Name"].unique()

<StringArray>
['ENGLAND', 'SCOTLAND']
Length: 2, dtype: str

In [8]:
countries.shape

(364, 18)

In [9]:
import pandas as pd

# Identify the population columns — the ones that start with "population_"
year_cols = [col for col in countries.columns if col.startswith("population_")]
print(f"Year columns to aggregate: {year_cols}")
print(f"Total year columns: {len(year_cols)}")

# Group by country, sum every year column across all ages and sexes
country_totals = (
    countries
    .groupby("Name")[year_cols]
    .sum()
    .reset_index()
)

print(f"\nShape after aggregation: {country_totals.shape}")
print(f"\nCountry totals per year:")
country_totals

Year columns to aggregate: ['population_2011', 'population_2012', 'population_2013', 'population_2014', 'population_2015', 'population_2016', 'population_2017', 'population_2018', 'population_2019', 'population_2020', 'population_2021', 'population_2022', 'population_2023', 'population_2024']
Total year columns: 14

Shape after aggregation: (2, 15)

Country totals per year:


,Name,population_2011,population_2012,population_2013,population_2014,population_2015,population_2016,population_2017,population_2018,population_2019,population_2020,population_2021,population_2022,population_2023,population_2024
0,ENGLAND,53107169,53506812,53918686,54370319,54808676,55289034,55619548,55924528,56230056,56325961,56554891,57144395,57932470,58620101
1,SCOTLAND,5299900,5308200,5316800,5331400,5350600,5373600,5388200,5392100,5411200,5408900,5412900,5447000,5506000,5546900


In [10]:
import pandas as pd

# Reshape from wide to long
population_lookup = country_totals.melt(
    id_vars="Name",
    value_vars=year_cols,
    var_name="year_col",
    value_name="population"
)

# Clean up: convert "population_2019" → 2019 as an integer
population_lookup["year"] = population_lookup["year_col"].str.replace("population_", "").astype(int)

# Standardise country name to match the "country" column in your combined dataset
population_lookup["country"] = population_lookup["Name"].str.title()

# Select only the columns we need
population_lookup = population_lookup[["country", "year", "population"]].sort_values(["country", "year"]).reset_index(drop=True)

print(f"Shape: {population_lookup.shape}")
print(f"Expected: 2 countries × 14 years = 28 rows")
print(f"\nSample rows:")
print(population_lookup.head(15))

Shape: (28, 3)
Expected: 2 countries × 14 years = 28 rows

Sample rows:
     country  year  population
0    England  2011    53107169
1    England  2012    53506812
2    England  2013    53918686
3    England  2014    54370319
4    England  2015    54808676
5    England  2016    55289034
6    England  2017    55619548
7    England  2018    55924528
8    England  2019    56230056
9    England  2020    56325961
10   England  2021    56554891
11   England  2022    57144395
12   England  2023    57932470
13   England  2024    58620101
14  Scotland  2011     5299900


In [12]:
import pandas as pd
from pathlib import Path

# Point at the two projection files — adjust filenames if yours differ
england_proj_file = Path("data/raw/ons/enpppsummary.xlsx")
scotland_proj_file = Path("data/raw/ons/scpppsummary.xlsx")

def get_projected_pop(file_path, country_name):
    """Pull mid-2025 and mid-2026 population from an ONS projection file."""
    df = pd.read_excel(file_path, sheet_name="PERSONS", skiprows=6)
    
    # Find the "Population at end" row
    pop_end_row = df[df["Components"] == "Population at end"].iloc[0]
    
    # Column names are strings, not integers
    pop_2025 = int(pop_end_row["2025"] * 1000)
    pop_2026 = int(pop_end_row["2026"] * 1000)
    
    return [
        {"country": country_name, "year": 2025, "population": pop_2025},
        {"country": country_name, "year": 2026, "population": pop_2026},
    ]

# Pull projections for both countries
projection_rows = get_projected_pop(england_proj_file, "England") + \
                  get_projected_pop(scotland_proj_file, "Scotland")

projections_df = pd.DataFrame(projection_rows)

print("Projected population 2025 & 2026:")
print(projections_df)

# Append to existing lookup
population_lookup = pd.concat([population_lookup, projections_df], ignore_index=True)
population_lookup = population_lookup.sort_values(["country", "year"]).reset_index(drop=True)

print(f"\nUpdated lookup shape: {population_lookup.shape}")
print(f"Expected: 2 countries × 16 years = 32 rows")
print(f"\nEngland rows:")
print(population_lookup[population_lookup["country"] == "England"])
print(f"\nScotland rows:")
print(population_lookup[population_lookup["country"] == "Scotland"])

Projected population 2025 & 2026:
    country  year  population
0   England  2025    58829244
1   England  2026    58951205
2  Scotland  2025     5549297
3  Scotland  2026     5546779

Updated lookup shape: (32, 3)
Expected: 2 countries × 16 years = 32 rows

England rows:
    country  year  population
0   England  2011    53107169
1   England  2012    53506812
2   England  2013    53918686
3   England  2014    54370319
4   England  2015    54808676
5   England  2016    55289034
6   England  2017    55619548
7   England  2018    55924528
8   England  2019    56230056
9   England  2020    56325961
10  England  2021    56554891
11  England  2022    57144395
12  England  2023    57932470
13  England  2024    58620101
14  England  2025    58829244
15  England  2026    58951205

Scotland rows:
     country  year  population
16  Scotland  2011     5299900
17  Scotland  2012     5308200
18  Scotland  2013     5316800
19  Scotland  2014     5331400
20  Scotland  2015     5350600
21  Scotland  2

In [13]:
population_lookup.to_csv("data/processed/population_lookup_2011_to_2026.csv", index=False)
print(f"Saved {len(population_lookup)} rows to data/processed/population_lookup_2011_to_2026.csv")

Saved 32 rows to data/processed/population_lookup_2011_to_2026.csv


In [14]:
import pandas as pd

# Load the combined dataset
combined = pd.read_csv("data/processed/combined_ae_2019_to_2026.csv", parse_dates=["period"])
print(f"Combined loaded: {len(combined):,} rows")

# Load the population lookup
population_lookup = pd.read_csv("data/processed/population_lookup_2011_to_2026.csv")
print(f"Population lookup loaded: {len(population_lookup):,} rows")

# Step 1 — extract year from period
combined["year"] = combined["period"].dt.year

# Step 2 — join population onto every row
combined_with_pop = combined.merge(
    population_lookup,
    on=["country", "year"],
    how="left"
)

print(f"\nAfter join: {len(combined_with_pop):,} rows")
print(f"Rows with missing population: {combined_with_pop['population'].isna().sum()}")

# Step 3 — calculate per-100k rates
combined_with_pop["attendances_per_100k"] = (
    combined_with_pop["attendances_type1"] / combined_with_pop["population"] * 100_000
)
combined_with_pop["breaches_4hr_per_100k"] = (
    combined_with_pop["breaches_4hr_type1"] / combined_with_pop["population"] * 100_000
)
combined_with_pop["waits_12hr_per_100k"] = (
    combined_with_pop["waits_12hr"] / combined_with_pop["population"] * 100_000
)

print(f"\nFinal shape: {combined_with_pop.shape}")
print(f"Columns: {list(combined_with_pop.columns)}")
print(f"\nSample rows:")
combined_with_pop[["country", "trust_name", "period", "population", 
                    "attendances_type1", "attendances_per_100k"]].head()

Combined loaded: 13,165 rows
Population lookup loaded: 32 rows

After join: 13,165 rows
Rows with missing population: 0

Final shape: (13165, 16)
Columns: ['trust_code', 'trust_region', 'trust_name', 'attendances_type1', 'breaches_4hr_type1', 'pct_4hr_type1', 'waits_12hr', 'admissions_type1', 'total_admissions', 'period', 'country', 'year', 'population', 'attendances_per_100k', 'breaches_4hr_per_100k', 'waits_12hr_per_100k']

Sample rows:


,country,trust_name,period,population,attendances_type1,attendances_per_100k
0,England,Basildon And Thurrock University Hospitals NHS...,2019-04-01,56230056,11354.0,20.192048
1,England,Bedford Hospital NHS Trust,2019-04-01,56230056,6349.0,11.291114
2,England,Cambridge University Hospitals NHS Foundation ...,2019-04-01,56230056,10412.0,18.516788
3,England,East And North Hertfordshire NHS Trust,2019-04-01,56230056,9167.0,16.302669
4,England,East Suffolk And North Essex NHS Foundation Trust,2019-04-01,56230056,16818.0,29.909271


In [15]:
combined_with_pop[["country", "trust_name", "period", "population", 
                    "attendances_type1", "attendances_per_100k"]].tail()

,country,trust_name,period,population,attendances_type1,attendances_per_100k
13160,Scotland,Balfour Hospital,2026-04-01,5546779,713.0,12.854307
13161,Scotland,Gilbert Bain Hospital,2026-04-01,5546779,804.0,14.494899
13162,Scotland,Ninewells Hospital,2026-04-01,5546779,5368.0,96.776886
13163,Scotland,Perth Royal Infirmary,2026-04-01,5546779,2159.0,38.923491
13164,Scotland,Western Isles Hospital,2026-04-01,5546779,554.0,9.987778


In [16]:
combined_with_pop.to_csv("data/processed/combined_ae_with_population.csv", index=False)
print(f"Saved {len(combined_with_pop):,} rows to data/processed/combined_ae_with_population.csv")

Saved 13,165 rows to data/processed/combined_ae_with_population.csv


In [17]:
import pandas as pd

# Load the combined dataset with population attached (but drop the misleading trust-level rates)
combined = pd.read_csv("data/processed/combined_ae_with_population.csv", parse_dates=["period"])
print(f"Loaded: {len(combined):,} rows")

# Drop the misleading trust-level per-100k columns we created earlier
combined = combined.drop(columns=[
    "attendances_per_100k",
    "breaches_4hr_per_100k",
    "waits_12hr_per_100k"
])
print(f"Dropped misleading trust-level per-100k columns")

# ============================================
# VIEW 1 — Country level (for Q2 benchmarking)
# ============================================

country_month = (
    combined
    .groupby(["country", "period", "year", "population"], as_index=False)
    .agg({
        "attendances_type1": "sum",
        "breaches_4hr_type1": "sum",
        "waits_12hr": "sum",
    })
)

# Recalculate 4-hour % from summed counts (weighted, not averaged)
country_month["pct_4hr_type1"] = (
    1 - (country_month["breaches_4hr_type1"] / country_month["attendances_type1"])
) * 100

# Per-100k rates using country population — honest, meaningful
country_month["attendances_per_100k"] = country_month["attendances_type1"] / country_month["population"] * 100_000
country_month["breaches_4hr_per_100k"] = country_month["breaches_4hr_type1"] / country_month["population"] * 100_000
country_month["waits_12hr_per_100k"] = country_month["waits_12hr"] / country_month["population"] * 100_000

print(f"\n=== VIEW 1: COUNTRY-LEVEL ===")
print(f"Shape: {country_month.shape}")
print(f"Expected: 2 countries × 85 months = 170 rows")
print(f"Sample:")
print(country_month.head(3))

# ============================================
# VIEW 2 — Region level using national population
# ============================================

region_month = (
    combined
    .groupby(["country", "trust_region", "period", "year", "population"], as_index=False)
    .agg({
        "attendances_type1": "sum",
        "breaches_4hr_type1": "sum",
        "waits_12hr": "sum",
    })
)

region_month["pct_4hr_type1"] = (
    1 - (region_month["breaches_4hr_type1"] / region_month["attendances_type1"])
) * 100

# Per-100k using NATIONAL population — note the honest column names
region_month["attendances_per_100k_national"] = region_month["attendances_type1"] / region_month["population"] * 100_000
region_month["breaches_4hr_per_100k_national"] = region_month["breaches_4hr_type1"] / region_month["population"] * 100_000
region_month["waits_12hr_per_100k_national"] = region_month["waits_12hr"] / region_month["population"] * 100_000

print(f"\n=== VIEW 2: REGION-LEVEL (national denominator) ===")
print(f"Shape: {region_month.shape}")
print(f"Unique regions per country:")
print(region_month.groupby("country")["trust_region"].nunique())
print(f"Sample:")
print(region_month.head(3))

# ============================================
# SAVE BOTH
# ============================================
country_month.to_csv("data/processed/country_month_view.csv", index=False)
region_month.to_csv("data/processed/region_month_view.csv", index=False)
print(f"\nSaved both views to data/processed/")

Loaded: 13,165 rows
Dropped misleading trust-level per-100k columns

=== VIEW 1: COUNTRY-LEVEL ===
Shape: (170, 11)
Expected: 2 countries × 85 months = 170 rows
Sample:
   country     period  year  population  attendances_type1  \
0  England 2019-04-01  2019    56230056          1330825.0   
1  England 2019-05-01  2019    56230056          1369332.0   
2  England 2019-06-01  2019    56230056          1334137.0   

   breaches_4hr_type1  waits_12hr  pct_4hr_type1  attendances_per_100k  \
0            304221.0       442.0      77.140420           2366.750266   
1            253619.0       414.0      81.478633           2435.231436   
2            250465.0       461.0      81.226441           2372.640354   

   breaches_4hr_per_100k  waits_12hr_per_100k  
0             541.029161             0.786056  
1             451.038142             0.736261  
2             445.429042             0.819846  

=== VIEW 2: REGION-LEVEL (national denominator) ===
Shape: (1785, 12)
Unique regions per cou

In [18]:
import pandas as pd

country_month = pd.read_csv("data/processed/country_month_view.csv", parse_dates=["period"])

print(f"Shape: {country_month.shape}")
print(f"Columns: {list(country_month.columns)}")
print(f"\nFirst 5 rows:")
country_month.head()

Shape: (170, 11)
Columns: ['country', 'period', 'year', 'population', 'attendances_type1', 'breaches_4hr_type1', 'waits_12hr', 'pct_4hr_type1', 'attendances_per_100k', 'breaches_4hr_per_100k', 'waits_12hr_per_100k']

First 5 rows:


,country,period,year,population,attendances_type1,breaches_4hr_type1,waits_12hr,pct_4hr_type1,attendances_per_100k,breaches_4hr_per_100k,waits_12hr_per_100k
0,England,2019-04-01,2019,56230056,1330825.0,304221.0,442.0,77.140420,2366.750266,541.029161,0.786056
1,England,2019-05-01,2019,56230056,1369332.0,253619.0,414.0,81.478633,2435.231436,451.038142,0.736261
2,England,2019-06-01,2019,56230056,1334137.0,250465.0,461.0,81.226441,2372.640354,445.429042,0.819846
3,England,2019-07-01,2019,56230056,1415918.0,264384.0,452.0,81.327732,2518.080366,470.182708,0.803841
4,England,2019-08-01,2019,56230056,1324070.0,254243.0,369.0,80.798372,2354.737118,452.147869,0.656233


In [19]:
region_month = pd.read_csv("data/processed/region_month_view.csv", parse_dates=["period"])

print(f"Shape: {region_month.shape}")
print(f"Columns: {list(region_month.columns)}")
print(f"\nUnique regions per country:")
print(region_month.groupby("country")["trust_region"].nunique())
print(f"\nFirst 10 rows:")
region_month.head(10)

Shape: (1785, 12)
Columns: ['country', 'trust_region', 'period', 'year', 'population', 'attendances_type1', 'breaches_4hr_type1', 'waits_12hr', 'pct_4hr_type1', 'attendances_per_100k_national', 'breaches_4hr_per_100k_national', 'waits_12hr_per_100k_national']

Unique regions per country:
country
England      7
Scotland    14
Name: trust_region, dtype: int64

First 10 rows:


,country,trust_region,period,year,population,attendances_type1,breaches_4hr_type1,waits_12hr,pct_4hr_type1,attendances_per_100k_national,breaches_4hr_per_100k_national,waits_12hr_per_100k_national
0,England,NHS England East Of England,2019-04-01,2019,56230056,144887.0,29652.0,13.0,79.534396,257.668248,52.733364,0.023119
1,England,NHS England East Of England,2019-05-01,2019,56230056,149136.0,23107.0,3.0,84.506088,265.224705,41.093681,0.005335
2,England,NHS England East Of England,2019-06-01,2019,56230056,147820.0,21601.0,18.0,85.386957,262.884319,38.415398,0.032011
3,England,NHS England East Of England,2019-07-01,2019,56230056,157046.0,25257.0,8.0,83.917451,279.291915,44.917259,0.014227
4,England,NHS England East Of England,2019-08-01,2019,56230056,147314.0,24192.0,8.0,83.577936,261.984445,43.023254,0.014227
5,England,NHS England East Of England,2019-09-01,2019,56230056,148971.0,25658.0,8.0,82.776514,264.931267,45.630401,0.014227
6,England,NHS England East Of England,2019-10-01,2019,56230056,148856.0,28457.0,10.0,80.882867,264.726750,50.608166,0.017784
7,England,NHS England East Of England,2019-11-01,2019,56230056,147129.0,31274.0,19.0,78.743823,261.655439,55.617942,0.033790
8,England,NHS England East Of England,2019-12-01,2019,56230056,149982.0,36297.0,29.0,75.799096,266.729238,64.550887,0.051574
9,England,NHS England East Of England,2020-01-01,2020,56325961,144138.0,32232.0,57.0,77.638097,255.899762,57.224057,0.101197


In [20]:
region_month.tail(10)

,country,trust_region,period,year,population,attendances_type1,breaches_4hr_type1,waits_12hr,pct_4hr_type1,attendances_per_100k_national,breaches_4hr_per_100k_national,waits_12hr_per_100k_national
1775,Scotland,NHS Western Isles,2025-07-01,2025,5549297,603.0,36.0,0.0,94.029851,10.866241,0.648731,0.000000
1776,Scotland,NHS Western Isles,2025-08-01,2025,5549297,623.0,59.0,0.0,90.529695,11.226647,1.063198,0.000000
1777,Scotland,NHS Western Isles,2025-09-01,2025,5549297,575.0,54.0,0.0,90.608696,10.361673,0.973096,0.000000
1778,Scotland,NHS Western Isles,2025-10-01,2025,5549297,503.0,43.0,0.0,91.451292,9.064211,0.774873,0.000000
1779,Scotland,NHS Western Isles,2025-11-01,2025,5549297,521.0,43.0,0.0,91.746641,9.388577,0.774873,0.000000
1780,Scotland,NHS Western Isles,2025-12-01,2025,5549297,506.0,31.0,0.0,93.873518,9.118272,0.558629,0.000000
1781,Scotland,NHS Western Isles,2026-01-01,2026,5546779,473.0,51.0,0.0,89.217759,8.527472,0.919453,0.000000
1782,Scotland,NHS Western Isles,2026-02-01,2026,5546779,419.0,46.0,1.0,89.021480,7.553934,0.829310,0.018028
1783,Scotland,NHS Western Isles,2026-03-01,2026,5546779,503.0,30.0,0.0,94.035785,9.068326,0.540854,0.000000
1784,Scotland,NHS Western Isles,2026-04-01,2026,5546779,554.0,22.0,0.0,96.028881,9.987778,0.396627,0.000000



DATA ENGINEERING — COMPLETE

Files on disk:
1. england_ae_2019_to_2026.csv (10,736 rows)
2. scotland_ae_2019_to_2026.csv (2,550 rows)
3. combined_ae_with_population.csv (13,165 rows) — trust level with population
4. country_month_view.csv (170 rows) — for Q2 benchmarking
5. region_month_view.csv (1,785 rows) — for regional analysis
6. population_lookup_2011_to_2026.csv (32 rows) — ONS denominators

Key decisions:
- Per-100k calculated only at country level (honest)
- Regional views use national population denominator (share of national load)
- Trust-level per-100k dropped (misleading without local catchment)
- Both countries aligned to April 2019 → April 2026 (85 months)
- Scotland at hospital site level; England at trust level (structural difference documented)
- 12-hour waits use Episode basis (Scotland) vs Attendance basis (England) — documented gap
- 2025 and 2026 population from ONS official projections

Next session: begin analysis with Q1 (performance trend).
